# Convert and Run Polyglot-Lion with OpenVINO™

[Polyglot-Lion](https://huggingface.co/knoveleng/polyglot-lion-0.6b) is a multilingual speech-recognition model based on [Qwen3-ASR](https://github.com/QwenLM/Qwen3-ASR). This tutorial converts the PyTorch model to stateful OpenVINO IR, compares recognition on the same audio, and benchmarks CPU or optional GPU inference. The compact 0.6B model is the default; the larger 1.7B checkpoint is optional.

**Experimental notebook:** the default run is a one-clip smoke test, not multilingual deployment validation. See [Conclusions](#Conclusions) and the [README](README.md) for scope and known limitations.

#### Table of contents:

- [Prerequisites](#Prerequisites)
    - [Install dependencies](#Install-dependencies)
- [Configure the tutorial](#Configure-the-tutorial)
    - [Select inference devices](#Select-inference-devices)
- [Prepare audio samples](#Prepare-audio-samples)
- [Export the stateful OpenVINO model](#Export-the-stateful-OpenVINO-model)
- [Load the PyTorch model](#Load-the-PyTorch-model)
- [Re-export the audio encoder](#Re-export-the-audio-encoder)
- [Compress model weights](#Compress-model-weights)
    - [Full INT8 weight compression](#Full-INT8-weight-compression)
    - [Mixed INT8/FP16 weight compression](#Mixed-INT8/FP16-weight-compression)
- [Validate OpenVINO model accuracy](#Validate-OpenVINO-model-accuracy)
- [Benchmark PyTorch and OpenVINO](#Benchmark-PyTorch-and-OpenVINO)
- [Interactive demo](#Interactive-demo)
- [Conclusions](#Conclusions)
- [Cleanup](#Cleanup)

### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

The first cell sets up a notebook-specific `.venv`. Keep `polyglot_helper.py` beside the notebook when testing an unmerged revision; the published notebook downloads the helper from this repository if it is missing. Git and pip are required to install the pinned exporter dependency.
<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/polyglot-lion/polyglot-lion.ipynb" />

## Prerequisites
[back to top ⬆️](#Table-of-contents:)

Use a Python 3.11–3.13 kernel with pip and Git. PyTorch runs on CPU; an XPU-enabled PyTorch wheel is not needed for OpenVINO GPU inference. Plan for at least 8 GB of system RAM for the default 0.6B checkpoint and 16 GB for the optional 1.7B checkpoint; these are planning estimates, not measured peak-memory guarantees. The model weights occupy about 1.58 GB (0.6B) or 4.08 GB (1.7B), before the environment, IR files and caches. Export is performed before the PyTorch reference is loaded to avoid running two full model loads at once. Conversion may take several minutes, depending on hardware.

The model-specific helper is loaded as described in Installation Instructions.

In [ ]:
import urllib.request
from pathlib import Path

# Helpers come from `utils` of the cloned repository, otherwise from GitHub; local copies are overwritten
helper_files = ["notebook_utils.py", "pip_helper.py"]
repo_utils = Path("../../utils")
in_repo = (repo_utils / "notebook_utils.py").is_file()
if in_repo:
    helper_contents = {helper_file: (repo_utils / helper_file).read_bytes() for helper_file in helper_files}
else:
    try:
        helper_contents = {}
        for helper_file in helper_files:
            url = f"https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/{helper_file}"
            with urllib.request.urlopen(url, timeout=10) as response:
                helper_contents[helper_file] = response.read()
    except OSError as error:
        if any(not Path(helper_file).is_file() for helper_file in helper_files):
            raise
        print(f"Could not update helper files, using the existing copies: {error}")
        helper_contents = {}

for helper_file, content in helper_contents.items():
    Path(helper_file).write_bytes(content)

from notebook_utils import setup_notebook_venv

# Requirements installed below go to a notebook-specific virtual environment in the `.venv` folder
setup_notebook_venv()

In [ ]:
import urllib.error
import urllib.request
from pathlib import Path

helper_path = Path("polyglot_helper.py")
if not helper_path.is_file():
    helper_url = "https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/notebooks/polyglot-lion/polyglot_helper.py"
    try:
        with urllib.request.urlopen(helper_url, timeout=10) as response:
            helper_contents = response.read()
    except urllib.error.URLError as error:
        raise RuntimeError("Could not download polyglot_helper.py. Place a copy from the notebook's revision beside it and retry.") from error
    helper_path.write_bytes(helper_contents)

### Install dependencies
[back to top ⬆️](#Table-of-contents:)

The first cell activates the notebook-specific `.venv`. Install through the shared `pip_helper` so packages are installed into that environment. Broad version ranges are used for runtime packages; the `qwen-asr` version and Optimum Intel commit are held because the encoder correction and export API were validated against those implementations. The Optimum Intel commit is from the same branch used by the Qwen3-ASR tutorial; this notebook pairs it with `qwen-asr==0.0.6` and Transformers 4.x, not the Transformers 5.x native Qwen3-ASR classes.

In [ ]:
from pip_helper import pip_install

# Avoid requiring the +cpu wheel tag, which is not used by macOS wheels.
pip_install(
    "-q",
    "--extra-index-url",
    "https://download.pytorch.org/whl/cpu",
    "torch>=2.6",
    "torchaudio>=2.6",
    "openvino>=2025.4.0",
    "nncf>=3.4.0",
    "qwen-asr==0.0.6",  # The encoder correction targets this reference implementation.
    "transformers>=4.57.6,<5",
    "datasets>=4.8.5",
    "numpy>=1.26,<3",
    "librosa>=0.10",
    "soundfile>=0.12",
    "pandas>=2.0",
    "ipywidgets>=8.0",
    "huggingface-hub>=0.30",
)

# Keep the verified exporter revision until the full-context encoder correction is upstreamed.
pip_install(
    "-q",
    "git+https://github.com/openvino-dev-samples/optimum-intel.git@4ca1144eafc3ef7d3d805a99c7b92953441437e5",
)

Enable the standard execution telemetry. To disable it,
set `SCARF_NO_ANALYTICS=1` before launching Jupyter.

In [ ]:
# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("polyglot-lion.ipynb")

## Configure the tutorial
[back to top ⬆️](#Table-of-contents:)

The default run uses the compact 0.6B checkpoint, CPU inference and one English clip. Optional compression and four-language evaluation are controlled here. The 1.7B checkpoint is available for comparison; newer checkpoint releases are not substituted without rechecking the encoder and compression behavior. Mixed INT8/FP16 compression is limited to the 1.7B variant because its accuracy policy was only validated there.

| Option | Purpose |
|---|---|
| `MODEL_VARIANT` | Choose `0.6B` (default) or the larger `1.7B` model. |
| `RUN_MIXED_INT8` | Build the experimentally validated mixed candidate for 1.7B only. |
| `RUN_INT8` | Build experimental full INT8 weights; accuracy is not established. |
| `RUN_MULTILINGUAL_EVALUATION` | Evaluate English, Mandarin, Tamil and Malay. |
| `MULTILINGUAL_SAMPLES_PER_LANGUAGE` | Set the size of that optional first-N slice. |
| `RUN_F32_DIAGNOSTICS` | Load a separate global-f32 model for logit diagnostics. |
| `MAX_AUDIO_SECONDS` | Reject longer local recordings before decoding them. |

GPU decoding requests f32 explicitly; the encoder keeps its task settings. Weight storage precision is not a guarantee of inference arithmetic.

In [ ]:
import json
from functools import partial
import shutil
import subprocess
import sys
from importlib.metadata import version
from pathlib import Path

import ipywidgets as widgets
import nncf
import openvino as ov
import pandas as pd
import torch
from IPython.display import Audio, display
from huggingface_hub import snapshot_download
from optimum.intel import OVModelForSpeechSeq2Seq
from qwen_asr.core.transformers_backend.modeling_qwen3_asr import (
    Qwen3ASRForConditionalGeneration,
)
from transformers import AutoProcessor, ProcessorMixin
from notebook_utils import device_widget
from polyglot_helper import (
    AUDIO_ENCODER_FIX,
    DATASET_ID,
    DATASET_REVISION,
    LANGUAGES,
    MAX_AUDIO_SECONDS,
    benchmark,
    configure_audio_encoder,
    corpus_error,
    export_audio_encoder,
    generate,
    load_samples,
    logit_metrics,
    model_fingerprint,
    IR_FILES,
    prepare_inputs,
    restore_decoder_projections,
    transcribe,
)

MODEL_OPTIONS = {
    "0.6B": (
        "knoveleng/polyglot-lion-0.6b",
        "801cdb314869f8f522afad0f476ffcdc7d9f5e39",
    ),
    "1.7B": (
        "knoveleng/polyglot-lion-1.7b",
        "9ef388b498182f50a5a7909e6e7a4e2d89a840c4",
    ),
}
MODEL_VARIANT = "0.6B"
MODEL_ID, MODEL_REVISION = MODEL_OPTIONS[MODEL_VARIANT]
RUN_INT8 = False
RUN_MIXED_INT8 = False
RUN_MULTILINGUAL_EVALUATION = False
MULTILINGUAL_SAMPLES_PER_LANGUAGE = 64
RUN_F32_DIAGNOSTICS = False
MAX_NEW_TOKENS = 512
ERROR_BUDGET = 0.02
CONFIG = {"PERFORMANCE_HINT": "LATENCY", "CACHE_DIR": ""}
GPU_DECODER_F32 = True
COMPONENT_CONFIGS = {"CPU": {}}
if GPU_DECODER_F32:
    COMPONENT_CONFIGS["GPU"] = {"decoder": {"INFERENCE_PRECISION_HINT": "f32"}}
if RUN_MIXED_INT8 and MODEL_VARIANT != "1.7B":
    raise ValueError("Mixed INT8/FP16 weights are only validated for MODEL_VARIANT='1.7B'.")
MODEL_DIR = Path("model") / MODEL_VARIANT.lower()
OUTPUT_DIR = Path("output") / MODEL_VARIANT.lower()
FP16 = MODEL_DIR / "fp16-full-context"
INT8 = MODEL_DIR / "int8-full-context"
MIXED = MODEL_DIR / "mixed-int8-first8-full-context"

### Select inference devices
[back to top ⬆️](#Table-of-contents:)

Choose CPU or an available Intel GPU with the widget below. The selected device is used for model evaluation, benchmarks and the interactive demo. The notebook always runs PyTorch reference inference on CPU.

In [ ]:
inference_device = device_widget(default="CPU", exclude=["NPU", "AUTO"])
display(inference_device)

The loader below configures the corrected audio encoder and applies any
decoder-only precision override **before compilation**. The same loader is
used for accuracy checks, benchmarks and the interactive demo.

In [ ]:
core = ov.Core()
versions = {
    name: version(name)
    for name in (
        "torch",
        "torchaudio",
        "openvino",
        "nncf",
        "transformers",
        "optimum-intel",
        "qwen-asr",
    )
}
if versions["qwen-asr"] != "0.0.6":
    raise RuntimeError("The encoder correction targets qwen-asr==0.0.6; install the listed dependencies.")
print("Model:", MODEL_ID, "at", MODEL_REVISION)
print("Versions:", versions)
print("Selected device:", inference_device.value)
print("Explicit component compile settings:", COMPONENT_CONFIGS)


def load_task_model(folder: Path, device: str, ov_config: dict, component_configs: dict) -> OVModelForSpeechSeq2Seq:
    """Apply the encoder adapter and device precision settings before compilation.

    :param folder: Directory containing the OpenVINO model files.
    :param device: OpenVINO device name used for compilation.
    :param ov_config: Compile configuration shared by the model components.
    :param component_configs: Additional per-component compile settings.
    :returns: A compiled speech-to-text model.
    """
    model = OVModelForSpeechSeq2Seq.from_pretrained(
        folder,
        device=device,
        trust_remote_code=True,
        ov_config=ov_config,
        compile=False,
    )
    configure_audio_encoder(model)
    for component, override in component_configs.get(device, {}).items():
        part = getattr(model, component)
        part.ov_config = {**part.ov_config, **override}
    model.compile()
    return model

## Prepare audio samples
[back to top ⬆️](#Table-of-contents:)

Download a deterministic test slice of [FLEURS](https://huggingface.co/datasets/google/fleurs) and prepare mono 16 kHz audio. The default is one English recording; enable the multilingual option for a larger slice. Model and dataset revisions are pinned, and cached audio hashes are verified.

The multilingual first-N slice is not speaker-disjoint. FLEURS is CC-BY-4.0, Google / Conneau et al. (2022), [paper](https://arxiv.org/abs/2205.12446). Polyglot-Lion is MIT and Qwen3-ASR is Apache-2.0; see the [source and license information](README.md).

In [ ]:
configs = tuple(LANGUAGES) if RUN_MULTILINGUAL_EVALUATION else ("en_us",)
sample_count = MULTILINGUAL_SAMPLES_PER_LANGUAGE if RUN_MULTILINGUAL_EVALUATION else 1
DATA_DIR = Path("data") / ("multilingual" if RUN_MULTILINGUAL_EVALUATION else "smoke")
samples = load_samples(configs=configs, count=sample_count, directory=DATA_DIR)
processor = AutoProcessor.from_pretrained(MODEL_ID, revision=MODEL_REVISION, trust_remote_code=True, fix_mistral_regex=True)
prepared = [
    prepare_inputs(
        processor=processor,
        audio_path=DATA_DIR / sample["audio"],
        max_duration_seconds=MAX_AUDIO_SECONDS,
    )
    for sample in samples
]
display(Audio(filename=str(DATA_DIR / samples[0]["audio"])))
print("Reference:", samples[0]["reference"])

## Export the stateful OpenVINO model
[back to top ⬆️](#Table-of-contents:)

Export before loading the PyTorch reference to avoid holding two source models in memory at once. The `automatic-speech-recognition-with-past` task exports a stateful text decoder that keeps its attention cache internally. The generation adapter resets this cache between utterances.

In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)
if FP16.exists() and not all((FP16 / name).is_file() for name in IR_FILES):
    raise ValueError("Incomplete export directory; choose a fresh model directory before exporting.")
if not FP16.exists():
    checkpoint = snapshot_download(repo_id=MODEL_ID, revision=MODEL_REVISION)
    exporter = shutil.which("optimum-cli", path=str(Path(sys.executable).parent))
    if exporter is None:
        raise FileNotFoundError("optimum-cli is missing from this kernel environment")
    subprocess.run(
        [
            exporter,
            "export",
            "openvino",
            "--model",
            checkpoint,
            "--task",
            "automatic-speech-recognition-with-past",
            "--trust-remote-code",
            "--weight-format",
            "fp16",
            str(FP16),
        ],
        check=True,
    )

## Load the PyTorch model
[back to top ⬆️](#Table-of-contents:)

Load the selected checkpoint on CPU and transcribe the prepared audio. PyTorch and OpenVINO use the same inputs, greedy decoding and token cap. Display the transcript and whether generation reached end-of-sequence (EOS).

In [ ]:
pt_model = Qwen3ASRForConditionalGeneration.from_pretrained(MODEL_ID, revision=MODEL_REVISION, trust_remote_code=True).eval()
pt_results = [
    transcribe(
        pt_model,
        processor=processor,
        inputs=inputs,
        backend="pytorch",
        max_new_tokens=MAX_NEW_TOKENS,
    )
    for inputs, _ in prepared
]
print("PyTorch:", pt_results[0]["transcription"])
print("EOS completion:", pt_results[0]["complete"])

Prepare one identical teacher-forced token prefix for the later raw-logit
comparison. This numerical diagnostic is separate from task-level WER/CER.

In [ ]:
# One identical teacher-forced prefix for raw-logit diagnostics.
logit_inputs = dict(prepared[0][0])
prefix = pt_results[0]["token_ids"][:8]
logit_inputs["input_ids"] = torch.cat([logit_inputs["input_ids"], torch.tensor([prefix], dtype=torch.long)], dim=1)
logit_inputs["attention_mask"] = torch.ones_like(logit_inputs["input_ids"])
with torch.inference_mode():
    pt_logits = pt_model.thinker(**logit_inputs, use_cache=False).logits[:, -1, :].float().numpy()

## Re-export the audio encoder
[back to top ⬆️](#Table-of-contents:)

After the PyTorch reference is loaded, re-export the audio encoder to match the pinned eager/SDPA implementation: full-utterance attention after removing CNN padding. The decoder export remains unchanged. This does not claim equivalence to Flash Attention's window semantics. Full-context attention has quadratic memory cost, so local WAV input is limited to 60 seconds.

In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)
export_source = {
    "model_id": MODEL_ID,
    "revision": MODEL_REVISION,
    "optimum_intel": versions["optimum-intel"],
    "openvino": versions["openvino"],
    "qwen_asr": versions["qwen-asr"],
    "audio_encoder_fix": AUDIO_ENCODER_FIX,
    "reference_encoder_attention": pt_model.thinker.audio_tower.config._attn_implementation,
}
export_record = FP16 / "export.json"
if not export_record.exists():
    export_audio_encoder(pt_model.thinker.audio_tower, FP16 / "openvino_encoder_model.xml")
    export_record.write_text(json.dumps({"source": export_source, "files": model_fingerprint(FP16)}, indent=2))
fp16_record = json.loads(export_record.read_text())
if fp16_record["source"] != export_source or fp16_record["files"] != model_fingerprint(FP16):
    raise ValueError("FP16 export changed; select a new output directory and export explicitly")

variants = {"FP16": FP16}

## Compress model weights
[back to top ⬆️](#Table-of-contents:)

Enable a compression option in the configuration cell to compare its model size and recognition accuracy against FP16.

### Full INT8 weight compression
[back to top ⬆️](#Table-of-contents:)

`RUN_INT8=True` compresses both encoder and decoder weights. Compare the per-language accuracy and EOS results with the FP16 baseline.

In [ ]:
if RUN_INT8:
    if not INT8.exists():
        INT8.mkdir(parents=True)
        for name in ("config.json", "generation_config.json"):
            shutil.copyfile(FP16 / name, INT8 / name)
        for component in ("encoder", "decoder"):
            filename = f"openvino_{component}_model.xml"
            compressed = nncf.compress_weights(
                core.read_model(FP16 / filename),
                mode=nncf.CompressWeightsMode.INT8_ASYM,
            )
            ov.save_model(compressed, INT8 / filename, compress_to_fp16=False)
            del compressed
        (INT8 / "compression.json").write_text(
            json.dumps(
                {
                    "source": model_fingerprint(FP16),
                    "nncf": versions["nncf"],
                    "mode": "INT8_ASYM",
                    "files": model_fingerprint(INT8),
                },
                indent=2,
            )
        )
    compression = json.loads((INT8 / "compression.json").read_text())
    if compression["source"] != model_fingerprint(FP16) or compression["nncf"] != versions["nncf"] or compression["files"] != model_fingerprint(INT8):
        raise ValueError("INT8 source/software/artifact changed; rebuild explicitly in a new directory")
    variants["INT8 weights"] = INT8

### Mixed INT8/FP16 weight compression
[back to top ⬆️](#Table-of-contents:)

`RUN_MIXED_INT8=True` keeps the corrected encoder in FP16 and compresses eligible decoder weights. The policy restores attention projections in all layers and all projections in the first eight layers. This option is restricted to the 1.7B checkpoint because only that model's compression policy was evaluated. GPU inference uses decoder-only f32 arithmetic by default; weight compression is not all-INT8 execution.

In [ ]:
if RUN_MIXED_INT8:
    mixed_source = {
        "fp16": model_fingerprint(FP16),
        "nncf": versions["nncf"],
        "policy": "attention-and-first8-fp16-v1",
        "first_n_layers": 8,
    }
    if not MIXED.exists():
        MIXED.mkdir(parents=True)
        for name in (
            "config.json",
            "generation_config.json",
            "openvino_encoder_model.xml",
            "openvino_encoder_model.bin",
        ):
            shutil.copyfile(FP16 / name, MIXED / name)
        filename = "openvino_decoder_model.xml"
        original_decoder = core.read_model(FP16 / filename)
        compressed_decoder = nncf.compress_weights(original_decoder.clone(), mode=nncf.CompressWeightsMode.INT8_ASYM)
        restored = restore_decoder_projections(compressed=compressed_decoder, reference=original_decoder, first_n_layers=8)
        if not restored:
            raise ValueError("No decoder projections were restored; inspect the exported graph.")
        ov.save_model(compressed_decoder, MIXED / filename, compress_to_fp16=False)
        del compressed_decoder, original_decoder
        (MIXED / "compression.json").write_text(
            json.dumps(
                {
                    "source": mixed_source,
                    "restored_projections": restored,
                    "files": model_fingerprint(MIXED),
                },
                indent=2,
            )
        )
    mixed_record = json.loads((MIXED / "compression.json").read_text())
    if mixed_record["source"] != mixed_source or mixed_record["files"] != model_fingerprint(MIXED):
        raise ValueError("Mixed model source/software/artifact changed; rebuild explicitly in a new directory")
    variants["Mixed INT8/FP16"] = MIXED

Compare the combined encoder and decoder BIN sizes for the enabled variants.
File size alone does not establish a compute-precision or performance benefit.

In [ ]:
sizes = {label: sum((folder / f"openvino_{part}_model.bin").stat().st_size for part in ("encoder", "decoder")) for label, folder in variants.items()}
display(pd.DataFrame([{"variant": label, "BIN MiB": size / 2**20} for label, size in sizes.items()]))

## Validate OpenVINO model accuracy
[back to top ⬆️](#Table-of-contents:)

Run the enabled variants on the same prepared inputs as PyTorch.
Compare raw logits using the identical teacher-forced prefix: shape/finite
checks, cosine similarity, RMSE, maximum absolute error and top-1 agreement
are diagnostics, not an `allclose` numerical certificate.

`RUN_F32_DIAGNOSTICS=True` loads a **separate** diagnostic model with a
global f32 hint; it does not change task evaluation or benchmark settings.

In [ ]:
devices = [inference_device.value]
results, numerical_rows = {}, []
for label, folder in variants.items():
    for device in devices:
        candidate = f"{label} {device}"
        model = load_task_model(folder, device=device, ov_config=CONFIG, component_configs=COMPONENT_CONFIGS)
        results[candidate] = [
            transcribe(
                model,
                processor=processor,
                inputs=inputs,
                backend="openvino",
                max_new_tokens=MAX_NEW_TOKENS,
            )
            for inputs, _ in prepared
        ]
        diagnostic_model = model
        diagnostic_config = dict(CONFIG)
        if RUN_F32_DIAGNOSTICS:
            diagnostic_config["INFERENCE_PRECISION_HINT"] = "f32"
            diagnostic_model = load_task_model(
                folder,
                device=device,
                ov_config=diagnostic_config,
                component_configs=COMPONENT_CONFIGS,
            )
        output = generate(
            diagnostic_model,
            inputs=logit_inputs,
            backend="openvino",
            max_new_tokens=1,
            return_dict_in_generate=True,
            output_logits=True,
        )
        numerical_rows.append(
            {
                "candidate": candidate,
                "diagnostic_config": diagnostic_config,
                "component_config": COMPONENT_CONFIGS[device],
                **logit_metrics(expected=pt_logits, actual=output.logits[0].float().numpy()),
            }
        )
        del diagnostic_model, model

Compare corpus WER for English, Tamil and Malay, and CER for Mandarin.
Scoring preserves Unicode combining marks and weights edit counts by
reference length. A local slice passes only if all outputs reach EOS and
each language stays within +2 percentage points of both PyTorch and the
same-device FP16 baseline.

In [ ]:
accuracy_rows = []
for candidate, outputs in results.items():
    device = candidate.rsplit(" ", 1)[1]
    for config in configs:
        indices = [i for i, sample in enumerate(samples) if sample["config"] == config]
        references = [samples[i]["reference"] for i in indices]
        characters = config == "cmn_hans_cn"
        pt_rate = corpus_error(
            references=references,
            hypotheses=[pt_results[i]["transcription"] for i in indices],
            characters=characters,
        )
        ov_rate = corpus_error(
            references=references,
            hypotheses=[outputs[i]["transcription"] for i in indices],
            characters=characters,
        )
        fp16_rate = corpus_error(
            references=references,
            hypotheses=[results[f"FP16 {device}"][i]["transcription"] for i in indices],
            characters=characters,
        )
        complete = all(outputs[i]["complete"] and pt_results[i]["complete"] for i in indices)
        accuracy_rows.append(
            {
                "candidate": candidate,
                "language": LANGUAGES[config],
                "metric": "CER" if characters else "WER",
                "PT %": pt_rate * 100,
                "OV %": ov_rate * 100,
                "delta PT pp": (ov_rate - pt_rate) * 100,
                "delta FP16 pp": (ov_rate - fp16_rate) * 100,
                "complete": complete,
                "local_slice_pass": complete and ov_rate <= pt_rate + ERROR_BUDGET + 1e-12 and ov_rate <= fp16_rate + ERROR_BUDGET + 1e-12,
            }
        )
display(pd.DataFrame(accuracy_rows))
display(pd.DataFrame(numerical_rows))

Save transcripts, generated tokens (including failures), sample provenance,
model fingerprints and compile settings to `output/<variant>/evaluation.json`.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "evaluation.json").write_text(
    json.dumps(
        {
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
            "dataset": DATASET_ID,
            "dataset_revision": DATASET_REVISION,
            "versions": versions,
            "compile_config": CONFIG,
            "component_compile_config": COMPONENT_CONFIGS,
            "audio_encoder_fix": AUDIO_ENCODER_FIX,
            "max_new_tokens": MAX_NEW_TOKENS,
            "samples": samples,
            "fingerprints": {label: model_fingerprint(folder) for label, folder in variants.items()},
            "pytorch": pt_results,
            "openvino": results,
            "accuracy": accuracy_rows,
            "numerical": numerical_rows,
        },
        ensure_ascii=False,
        indent=2,
    )
)

## Benchmark PyTorch and OpenVINO
[back to top ⬆️](#Table-of-contents:)

Warm up each model and compare median transcription latency on the same audio. PyTorch runs on CPU; OpenVINO uses the selected device. The table also reports whether repeated runs generated identical tokens.

In [ ]:
devices = [inference_device.value]
first_inputs = prepared[0][0]
timings = [
    {
        "candidate": "PyTorch CPU",
        **benchmark(
            lambda model=pt_model: transcribe(
                model,
                processor=processor,
                inputs=first_inputs,
                backend="pytorch",
                max_new_tokens=MAX_NEW_TOKENS,
            )
        ),
    }
]
for label, folder in variants.items():
    for device in devices:
        model = load_task_model(folder, device=device, ov_config=CONFIG, component_configs=COMPONENT_CONFIGS)
        timings.append(
            {
                "candidate": f"{label} {device}",
                **benchmark(
                    lambda model=model: transcribe(
                        model,
                        processor=processor,
                        inputs=first_inputs,
                        backend="openvino",
                        max_new_tokens=MAX_NEW_TOKENS,
                    )
                ),
            }
        )
        del model
display(pd.DataFrame(timings))
(OUTPUT_DIR / "performance.json").write_text(
    json.dumps(
        {
            "versions": versions,
            "compile_config": CONFIG,
            "component_compile_config": COMPONENT_CONFIGS,
            "audio_encoder_fix": AUDIO_ENCODER_FIX,
            "fingerprints": {label: model_fingerprint(folder) for label, folder in variants.items()},
            "warmup": 1,
            "iterations": 3,
            "sample": samples[0],
            "results": timings,
        },
        indent=2,
    )
)

## Interactive demo
[back to top ⬆️](#Table-of-contents:)

Select an enabled weight variant and use the shared inference-device selector. Enter a local WAV path or leave it blank to use the prepared sample. Audio is resampled to mono 16 kHz; recordings longer than 60 seconds are rejected before decoding. The compiled demo model is cached for each variant/device selection.

In [ ]:
variant_widget = widgets.Dropdown(options=list(variants), description="Weights:")
audio_widget = widgets.Text(value="", description="Local WAV:", placeholder="Optional path; maximum 60 seconds")
button, demo_output = widgets.Button(description="Transcribe"), widgets.Output()
demo_model_cache = {}


def transcribe_selection(
    audio_path: str,
    variant_name: str,
    selected_device: str,
    variants: dict[str, Path],
    processor: ProcessorMixin,
    default_sample: Path,
    max_new_tokens: int,
    max_audio_seconds: float,
    model_cache: dict,
    ov_config: dict,
    component_configs: dict,
) -> dict:
    """Transcribe the selected WAV with a cached OpenVINO model.

    :param audio_path: Optional local WAV path; empty selects the prepared sample.
    :param variant_name: Enabled OpenVINO weight variant.
    :param selected_device: Device selected by the tutorial device widget.
    :param variants: Mapping from displayed variant names to IR directories.
    :param processor: Processor shared by the PyTorch and OpenVINO paths.
    :param default_sample: Audio sample used when no local path is entered.
    :param max_new_tokens: Generation token limit.
    :param max_audio_seconds: Maximum accepted WAV duration.
    :param model_cache: Per-variant/device cache of compiled demo models.
    :param ov_config: Compile configuration shared by model components.
    :param component_configs: Per-component compile settings.
    :returns: Transcription and generation metadata.
    """
    selected_path = Path(audio_path).expanduser() if audio_path.strip() else default_sample
    inputs, _ = prepare_inputs(
        processor=processor,
        audio_path=selected_path,
        max_duration_seconds=max_audio_seconds,
    )
    cache_key = (variant_name, selected_device)
    if cache_key not in model_cache:
        model_cache[cache_key] = load_task_model(
            folder=variants[variant_name],
            device=selected_device,
            ov_config=ov_config,
            component_configs=component_configs,
        )
    result = transcribe(
        model_cache[cache_key],
        processor=processor,
        inputs=inputs,
        backend="openvino",
        max_new_tokens=max_new_tokens,
    )
    if not result["complete"]:
        raise RuntimeError("Transcription did not reach EOS within the token budget")
    return result


def display_transcription(
    _: widgets.Button,
    *,
    output: widgets.Output,
    audio_path: widgets.Text,
    variant: widgets.Dropdown,
    inference_device: widgets.Dropdown,
    variants: dict[str, Path],
    processor: ProcessorMixin,
    default_sample: Path,
    max_new_tokens: int,
    max_audio_seconds: float,
    model_cache: dict,
    ov_config: dict,
    component_configs: dict,
) -> None:
    """Run the interactive transcription and render callback errors visibly.

    :param output: Widget output area for transcript or error text.
    :param audio_path: Widget containing an optional local audio path.
    :param variant: Weight-variant selector.
    :param inference_device: Shared device selector used by evaluation and the demo.
    :param variants: Mapping from displayed variant names to IR directories.
    :param processor: Model processor.
    :param default_sample: Prepared audio used for an empty path.
    :param max_new_tokens: Generation token limit.
    :param max_audio_seconds: Maximum accepted WAV duration.
    :param model_cache: Per-variant/device cache of compiled demo models.
    :param ov_config: Compile configuration shared by model components.
    :param component_configs: Per-component compile settings.
    :returns: None.
    """
    output.clear_output(wait=False)
    with output:
        try:
            result = transcribe_selection(
                audio_path=audio_path.value,
                variant_name=variant.value,
                selected_device=inference_device.value,
                variants=variants,
                processor=processor,
                default_sample=default_sample,
                max_new_tokens=max_new_tokens,
                max_audio_seconds=max_audio_seconds,
                model_cache=model_cache,
                ov_config=ov_config,
                component_configs=component_configs,
            )
            print(result["transcription"])
        except Exception as error:
            print(f"Transcription failed: {type(error).__name__}: {error}")


button.on_click(
    partial(
        display_transcription,
        output=demo_output,
        audio_path=audio_widget,
        variant=variant_widget,
        inference_device=inference_device,
        variants=variants,
        processor=processor,
        default_sample=DATA_DIR / samples[0]["audio"],
        max_new_tokens=MAX_NEW_TOKENS,
        max_audio_seconds=MAX_AUDIO_SECONDS,
        model_cache=demo_model_cache,
        ov_config=CONFIG,
        component_configs=COMPONENT_CONFIGS,
    )
)
display(variant_widget, audio_widget, button, demo_output)

## Conclusions
[back to top ⬆️](#Table-of-contents:)

Use the accuracy and timing tables above to inspect the exact model revision, audio slice, device and software configuration run in this notebook. The default evaluates one English clip; enable the multilingual option for a broader, but still small and non-speaker-disjoint, FLEURS slice. WER/CER on a small slice is not a deployment evaluation.

The corrected encoder follows the eager/SDPA full-context behavior of the pinned reference implementation. The 60-second local-audio limit bounds the quadratic attention path. The optional compression variants are exploratory comparisons; a local accuracy pass does not establish multilingual robustness or deployment readiness. Interpret compression results alongside the accuracy and timing outputs produced for the selected model, data slice and device.

The tutorial demonstrates conversion, weight compression, task-level accuracy comparison and end-to-end generation timing. It does not certify performance on every device, strict raw-logit equivalence, speaker-disjoint generalization, or deployment quality. See the [README](README.md) for sources and licensing.

## Cleanup
[back to top ⬆️](#Table-of-contents:)

The optional commands below remove only this tutorial's generated models,
audio and reports. Keep them commented unless these files are no longer
needed. To reset dependencies, stop the kernel before deleting `.venv`.
Do not remove shared download caches.

In [ ]:
# shutil.rmtree("model")
# shutil.rmtree("data")
# shutil.rmtree("output")
# Path("notebook_utils.py").unlink(missing_ok=True)
# del pt_model  # Optional: run after all comparisons and benchmarks to release reference-model memory.